# Demographic analysis

This notebook identifies correlations with demographic features and arguments brought forward by the advertisements.

## Setup

In [ ]:
from hydra import compose, initialize
from omegaconf import DictConfig

with initialize(config_path="configs", job_name="dem_analysis", version_base="1.3"):
    cfg: DictConfig = compose(config_name="config")

Load and merge dataframes

In [ ]:
import pandas as pd

df = pd.read_csv(cfg.data.path)
df = df[df["cluster"] != -1]

df_summaries = pd.read_csv(cfg.data.annotation_path)
df = df.merge(df_summaries, left_on="cluster", right_index=True)
df["stance_topic"] = df["stance"] + " " + df["topic"]

df.head()

# Topic Analysis

In [ ]:
# fmt: off
us_states = [
    'Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'Florida',
    'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine',
    'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada',
    'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma',
    'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah',
    'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming'
]

us_states_to_abbrev = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA', 'Colorado': 'CO',
    'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID',
    'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA',
    'Maine': 'ME', 'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH', 'Oklahoma': 'OK',
    'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC', 'South Dakota': 'SD',
    'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA',
    'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}
# fmt: on

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns


def ppmi(
    target="topic",
    target_subset=None,
    genders: list[str] | None = None,
    ages: list[str] | None = None,
    states: list[str] | None = None,
):
    """Compute the PMI between topics and demographics.

    Parameters:
        target (str): The column to use as the target variable. Default is 'topic'.
        target_subset (list): A subset of the target variable to display.
        genders, ages, states (list): Which granularities to include. Default is all.

    Returns:
        pd.DataFrame: A DataFrame of PMI values between topics and corresponding demographics.
    """
    # Initialize Default Demographics
    genders = genders or ["male", "female"]
    ages = ages or ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]
    states = states or []

    # Create Demographic Columns
    if ages and genders:
        demographics = [f"{gender}_{age}" for age in ages for gender in genders]
    else:
        demographics = genders or ages or states

    # Create DataFrame for PPMI Calculation
    states = [state + "%" for state in states]
    temp = df[[target] + demographics + states].copy()

    # Interaction Terms
    if states and (ages or genders):
        for col in demographics:
            for state in states:
                temp[f"{col}_{state}"] = temp[col] * temp[state]
        demographics = [f"{col}_{state}" for col in demographics for state in states]

    # Melt DataFrame to Long Format
    df_long = temp.melt(id_vars=target, value_vars=demographics, var_name="demographic", value_name="views")

    # Calculate Probabilities
    total_views = df["avg_impressions"].sum()
    P_g = df.groupby(target)["avg_impressions"].sum() / total_views  # P(topic)
    P_d = df_long.groupby("demographic")["views"].sum() / total_views  # P(demographic)
    P_gd = df_long.groupby([target, "demographic"])["views"].sum() / total_views  # P(topic, demo)

    # Calculate PPMI
    topics = [t for t in df[target].unique() if t != "sewer sale"]
    if target_subset:
        topics = [t for t in topics if t in target_subset]
    PPMI = pd.DataFrame(index=topics, columns=demographics)

    # Fill PPMI DataFrame
    for g in topics:
        for d in demographics:
            if P_gd[(g, d)] == 0 or P_g[g] == 0 or P_d[d] == 0:
                ppmi = 0
            else:
                ppmi = max(0, np.log(P_gd[(g, d)] / (P_g[g] * P_d[d])))
            PPMI.loc[g, d] = ppmi

    return PPMI.astype(float)


def visualize_topic(**kwargs):
    """Visualize the PMI between topics and demographics as a heatmap."""
    PPMI = ppmi(**kwargs)
    plt.figure(figsize=(12, 6))
    annot = PPMI.map(lambda x: f"{x:.2f}" if x != 0 else "")
    sns.heatmap(PPMI, cmap="coolwarm", center=0, annot=annot, fmt="", vmax=2)
    plt.title("PPMI: Topic vs. Demographic Overrepresentation")
    plt.xlabel("Demographic")
    plt.ylabel("Topic")
    plt.tight_layout()
    plt.show()

# Usage

In [ ]:
visualize_topic(
    # target='stance_topic',
    # target='topic',
    target="moral_foundation",
    # target='stance',
    genders=["female"],
    # genders=['male'],
    # genders=None,
    ages=["18-24", "65+"],
    # ages=None,
    # states=['California', 'Texas', 'New York', 'Florida', 'Pennsylvania', 'Ohio']
)